In [2]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, "..")
from importlib import reload

import equations as eq
import trajectory_lib as tr
reload (tr);
reload (eq);

motion_list  =  ['drinking'] #'Elevation_yzy','Scabduction_yzy',
participant = '1LU'
GH_seq = 'YZY' 
weight= 101
include_q_clavicula_init = False

OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model.mat')
include_activation_dynamics = False

act_w = 1
w_diff_vel = 1

MM,FO,q,u,fr,frstar,kindeq,xdot,first_elips_scale,elips_trans = eq.create_eoms_eul(OS_struct,derive = 'numeric',gen_matlab_functions = 0,GH_seq = GH_seq)
TE,activations,TE_conoid = eq.polynomials_euler(OS_struct,q,u,derive = 'numeric')


dimensions: [0.20506566 0.32022407 0.13557591]
trans [ 0.         -0.21570185  0.08806762]
equations created
[ 0.1881 -0.0114 -0.01  ]
[-0.005   -0.02226 -0.03224]


In [6]:
for imot in range(len(motion_list)):
    if include_q_clavicula_init:
        q_clavicula_init = sc.io.loadmat('../Motions/'+participant+'/'+motion_list[imot]+'/res_euler_'+motion_list[imot]+'_100.mat')['data']['q_clavicula_init']

    traj_w = weight
    struct_name = 'res_euler_'+motion_list[imot]+'_'+str(weight)
    eoms_implicit = sp.Matrix(kindeq).col_join(fr+frstar+TE+sp.Matrix(TE_conoid))

    if include_activation_dynamics:
        excitations = []
        act_ode = []
        # print(activations)
        for i in range(len(activations)):
            excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
            act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))

        sp_act_ode = sp.Matrix(act_ode)
        eoms_implicit = eoms_implicit.col_join(sp_act_ode)
    reload(eq);
    num_nodes = 101
    file = '../Motions/'+participant+'/' + motion_list[imot] + '/' + motion_list[imot]
    traj_original, interval_value, time = tr.exp_trajectory_eul(file,num_nodes)
    traj = tr.exp_trajectory_eul_myobj(traj_original,GH_seq)

    if include_activation_dynamics:
        state_symbols = tuple(q+u+activations)
        specified_symbols = tuple(excitations)
    else:
        state_symbols = tuple(q+u)
        specified_symbols = tuple(activations)

    num_states = len(state_symbols)
    num_q = len(q)
    num_u = len(u)

    num_inputs = len(specified_symbols)
    t = me.dynamicsymbols._t
    objective_traj,objective_traj_jac = eq.custom_objective_eul(len(q),interval_value, GH_seq = GH_seq)
    obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes, interval_value)
    w_diff_act = 0
    w_diff_exc = 1e-1
    node1 = 0
    node2 = num_nodes//2
    node3 = num_nodes-1

    def obj(free):
        # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:10*num_nodes])**2)
        min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
        # min_traj = traj_w * (objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3]))

        min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
        min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
        # min_act_dif = w_diff_act * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))))
        if include_activation_dynamics:
            min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))

        # min_act_dif = obj_act_dif(np.ones((101,5)))
        obj = (min_traj + min_torque + min_vel_dif) # 

        if include_activation_dynamics:
            obj += min_exc_dif
            
        return obj.item() # + min_vel_dif + min_act_dif + min_exc_dif

    def obj_grad(free):
        grad = np.zeros_like(free)
        # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj_original.flatten())
        grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))

        grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] #+ w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
        if include_activation_dynamics:
            grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] = w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

        grad[num_q*num_nodes:(num_q + num_u)*num_nodes] = w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

        # ## reach ##
        # first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1])))
        # second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2])))
        # third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3])))
        # # print(np.shape(first_grad_vals))
        # grad_traj = np.zeros((num_nodes,10))
        # grad_traj[node1,:] = first_grad_vals
        # grad_traj[node2,:] = second_grad_vals
        # grad_traj[node3,:] = third_grad_vals
        # grad[:10*num_nodes] = traj_w * np.concatenate(grad_traj.T)
        # ## end reach ##

        return grad
    instance_constraints = []
    if include_q_clavicula_init:
        instance_constraints.append(state_symbols[0].func(0.0)-q_clavicula_init[0,0][0,0])
        instance_constraints.append(state_symbols[1].func(0.0)-q_clavicula_init[0,0][0,1])
        instance_constraints.append(state_symbols[2].func(0.0)-q_clavicula_init[0,0][0,2])
    instance_constraints.append(state_symbols[-1].func(0.0)-0)

    bounds1 = (0.0,1.0)
    bounds = (bounds1,)*len(activations)
    bndrs = dict(zip(activations,bounds))
    if include_activation_dynamics:
        bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
        bndrs.update(bndrs_exc)

    for i in range(num_q):
        bndrs.update({q[i]: (min(traj_original[i,:])-0.2, max(traj_original[i,:])+0.2)})

    # angles_bndr = {q[0]: (-1.2, -0.6),
    #            q[1]: (-0.0, 0.3),
    #            q[2]: (-20*np.pi/180, 50*np.pi/180),
    #            q[3]: (0.6, 1.1),
    #            q[4]: (-0.3, 0.3),
    #            q[5]: (-0.3,0.3),
    #            q[9]: (0*np.pi/180,55*np.pi/180)}
    # bndrs.update(angles_bndr)

    start = tm.time()
    prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                num_nodes, interval_value,
                known_parameter_map={},
                instance_constraints=instance_constraints,
                bounds=bndrs,
                integration_method='midpoint',
    ) #               
    time_to_create = tm.time() - start
    print(time_to_create)
    prob.add_option('max_iter',10000)
    prob.add_option('limited_memory_max_history', 40)
    reload(tr);
    initial_guess = np.zeros(prob.num_free)
    initial_guess[:10*num_nodes] = traj_original.flatten()
    # initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+motion_folder+'/res_euler_Flexion_yzy_100.mat',prob.num_free)
    # initial_guess = tr.initial_guess_from_solution('../Motions/'+motion_folder+'/res_euler_Scabduction_yzy_scaled_noexc_81.mat',prob.num_free)

    time_2_solve_start = tm.time()
    solution, info = prob.solve(initial_guess)
    time_2_solve = tm.time() - time_2_solve_start
    print(info['status_msg'])
    print(info['obj_val'])
    act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
    objective_value = prob.obj_value
    print('Objective activations: ', act_obj)
    reload(tr)

    file_name = '../Motions/'+participant+'/'+motion_list[imot]+'/' + struct_name + '.mat'
    tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

    file_name_mot = '../Motions/'+participant+'/'+motion_list[imot]+'/' + struct_name + '.mot'
    tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot,GH_seq)

816.6789600849152
This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:   588001
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    14847
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    13837
                     variables with only upper bounds:        0
Total number of equality constraints.................:     2001
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  6.0286779e-02 1.76e+03 1.46e-03   0.0 0.00e+00    -  

KeyboardInterrupt: 

In [4]:
# from opty import Problem, create_objective_function, parse_free
# import sympy as sp
# import numpy as np
# import scipy as sc
# import time as tm
# import pickle
# import sympy.physics.mechanics as me
# import sys
# sys.path.insert(0, "..")
# from importlib import reload
# import matplotlib.pyplot as plt
# import equations as eq
# reload (eq);
# import trajectory_lib as tr
# reload (tr);

# participant = '1LU'
# motion_list  = ['Scabduction_yzy','Flexion_yzy']
# GH_seq = 'YZY'

# OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model.mat')
# include_activation_dynamics = True
# include_q_clavicula_init = True

# act_w = 1

# MM,FO,q,w,u0,fr,frstar,kinematical,xdot,holonomic,first_elips_scale,elips_trans = eq.create_eoms_u0state(OS_struct,derive = 'numeric',gen_matlab_functions = 0)
# TE,activations,TE_conoid = eq.polynomials_quat(model_struct = OS_struct,q = q,u = w, derive = 'numeric')
# reload(eq)
# reload(tr)
# clav_pos = [0.4]
# weights = [230]


In [5]:

# for imot in range(len(motion_list)):
#     for ipos in range(len(clav_pos)):
#         if include_q_clavicula_init:
#             q_clavicula_init = sc.io.loadmat('../Motions/'+participant+'/'+motion_list[imot]+'/res_quat_'+motion_list[imot]+'_204.mat')['data']['q_clavicula_init']

#         weight = int(weights[0] + clav_pos[ipos]*10)
#         print(weight)
#         struct_name = 'res_quat_'+motion_list[imot]+'_'+str(weight)
#         traj_w = weight
#         excitations = []
#         act_ode = []
#         if include_activation_dynamics:
#             for i in range(len(activations)):
#                 excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
#                 act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))
#                 sp_act_ode = sp.Matrix(act_ode)
                
#             eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+TE+sp.Matrix(TE_conoid)).col_join(holonomic).col_join(sp_act_ode)

#         else:
#             eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+TE+sp.Matrix(TE_conoid)).col_join(holonomic)

#         reload(eq)
#         num_nodes = 101
#         file = '../Motions/' + participant + '/' + motion_list[imot] + '/' + motion_list[imot]
#         traj_original, interval_value, time = tr.exp_trajectory_quat(file,num_nodes)
#         traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos[ipos])
        
#         if include_activation_dynamics:
#             state_symbols = tuple(q+w+u0+activations)
#             specified_symbols = tuple(excitations)
#         else:
#             state_symbols = tuple(q+w+u0)
#             specified_symbols = tuple(activations)

#         num_states = len(state_symbols) 
#         num_q = len(q)
#         num_u = len(w+u0)
#         num_inputs = len(specified_symbols)
#         t = me.dynamicsymbols._t
#         objective_traj,objective_traj_jac = eq.custom_objective_quat(num_q,interval_value,clav_pos[ipos])
#         obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes, interval_value)
#         w_diff_vel = 1
#         w_diff_act = 0
#         w_diff_exc = 1e-1
#         node1 = 0
#         node2 = num_nodes//2
#         node3 = num_nodes-1

#         def obj(free):
#             # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:10*num_nodes])**2)
#             min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
#             # min_traj = traj_w * (objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3]))

#             min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
#             min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
            
#             # min_act_dif = w_diff_act * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))))
#             if include_activation_dynamics:
#                 min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))
            
#             obj = (min_traj + min_torque + min_vel_dif)
#             if include_activation_dynamics:
#                 obj += min_exc_dif

#             return obj.item()

#         def obj_grad(free):
#             grad = np.zeros_like(free)
#             # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj_original.flatten())
#             grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))

            
#             grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] #+ w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
#             if include_activation_dynamics:
#                 grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] = w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))
#             grad[num_q*num_nodes:(num_q + num_u)*num_nodes] = w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

#             # ## reach ##
#             # first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1])))
#             # second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2])))
#             # third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3])))
#             # # print(np.shape(first_grad_vals))
#             # grad_traj = np.zeros((num_nodes,10))
#             # grad_traj[node1,:] = first_grad_vals
#             # grad_traj[node2,:] = second_grad_vals
#             # grad_traj[node3,:] = third_grad_vals
#             # grad[:10*num_nodes] = traj_w * np.concatenate(grad_traj.T)
#             # ## end reach ##

#             return grad
#         instance_constraints = []
#         # for i in range(13):
#         if include_q_clavicula_init:
#             instance_constraints.append(state_symbols[0].func(0.0)-q_clavicula_init[0,0][0,0])
#             instance_constraints.append(state_symbols[1].func(0.0)-q_clavicula_init[0,0][0,1])
#             instance_constraints.append(state_symbols[2].func(0.0)-q_clavicula_init[0,0][0,2])
#             instance_constraints.append(state_symbols[3].func(0.0)-q_clavicula_init[0,0][0,3])
#         instance_constraints.append(state_symbols[-1].func(0.0)-0) 
            
#         bounds1 = (0.0,1.0)
#         bounds = (bounds1,)*len(activations)
#         bndrs = dict(zip(activations,bounds))
#         if include_activation_dynamics:
#             bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
#             bndrs.update(bndrs_exc)

#         SCq_bndr = {q[0] : (0.85, 1),
#                     q[1] : (-0.1, 0.2),
#                     q[2] : (-0.5, -0.25),
#                     q[3] : (-0.1, 0.25)}
#         bndrs.update(SCq_bndr)
#         ACq_bndr = {q[4] : (0.85, 1),
#                     q[5] : (-0.1, 0.2),
#                     q[6] : (0.3, 0.5),
#                     q[7] : (-0.1, 0.1)}
#         bndrs.update(ACq_bndr)
#         GHq_bndr = {q[8] : (0.8, 1),
#                     q[12]: (00*np.pi/180, 55*np.pi/180)}
#         bndrs.update(GHq_bndr)
#         print(bndrs)


#         start = tm.time()
#         prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
#                     num_nodes, interval_value,
#                     known_parameter_map={},
#                     instance_constraints=instance_constraints,
#                     bounds=bndrs,
#                     integration_method='midpoint')


#         time_to_create = tm.time() - start
#         print(time_to_create)



#         prob.add_option('max_iter',10000)
#         prob.add_option('limited_memory_max_history', 40)
#         initial_guess = np.zeros(prob.num_free)
#         initial_guess[:13*num_nodes] = traj_original.flatten()
#         # initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+motion_folder+'/res_quat_Elevation_yzy_103.mat',prob.num_free)
#         time_2_solve_start = tm.time()
#         solution, info = prob.solve(initial_guess)
#         time_2_solve = tm.time() - time_2_solve_start
#         print(info['status_msg'])
#         print(info['obj_val'])
#         act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
#         objective_value = prob.obj_value
#         print('Objective activations: ', act_obj)

#         reload(tr)
#         file_name = '../Motions/'+participant+'/'+motion_list[imot]+'/' + struct_name + '.mat'
#         tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

#         file_name_mot = '../Motions/'+participant+'/'+motion_list[imot]+'/' + struct_name + '.mot'
#         tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)